# Lecture 09: Web Data Extraction — Solutions

This notebook contains working solutions for all exercises. Run each cell in order.

In [ ]:
import requests
from bs4 import BeautifulSoup
from requests.exceptions import RequestException
import urllib.parse
from IPython.display import SVG, display

## Part 1: The `requests` library

### GET request & status code

In [ ]:
res = requests.get('https://sparr.chemie.unibas.ch/en/teaching/')
print('Status code:', res.status_code)   # 200 = success

### Accessing the response text and headers

In [ ]:
print(res.text[:500])   # first 500 chars of raw HTML

In [ ]:
print(res.headers)

### Error handling with try/except

In [ ]:
url = 'https://httpbin.org/get'

try:
    response = requests.get(url)
    response.raise_for_status()
except RequestException as err:
    print(f'An Error Occurred: {err}')
else:
    print(response.text)

## Part 2: BeautifulSoup basics

In [ ]:
soup = BeautifulSoup(res.text, 'html.parser')

# Navigate with .contents
first_link = soup.a
print('First link contents:', first_link.contents)

# Find all <a> tags
all_links = soup.find_all('a')
print(f'Total <a> tags: {len(all_links)}')

# Access href attribute
print('First href:', soup.a['href'])

## Question 1 — Extract all URLs from example.com

In [ ]:
url = 'https://example.com'
response = requests.get(url)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    links = soup.find_all('a')
    for link in links:
        href = link.get('href')
        if href:
            print(href)
else:
    print('Webpage retrieval failed.')

## Question 2 — Find all exercise & solution PDFs from the Sparr Group page

Inspecting the page HTML reveals that all the teaching PDFs live inside `<div id="c1042">`. We can target that div directly and collect every `<a>` tag whose `href` ends in `.pdf`.

Expected result: **577 links** (as of 2026).

In [ ]:
url = 'https://sparr.chemie.unibas.ch/en/teaching/'
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

# All teaching PDFs are inside div#c1042
div = soup.find('div', id='c1042')
anchor_tags = div.find_all('a')

pdf_links = []
for anchor in anchor_tags:
    href = anchor.get('href', '')
    if href.endswith('.pdf'):
        if not href.startswith('http'):
            href = 'https://sparr.chemie.unibas.ch' + href
        pdf_links.append(href)

print(f'Found {len(pdf_links)} PDF links')
print('First 5:')
for link in pdf_links[:5]:
    print(' ', link)

## Question 3 — Chatbot-generated solution

Ask Claude or ChatGPT to write the scraper by pasting the relevant HTML snippet.
A typical prompt: *"Here is HTML from a teaching page. Find all PDF links inside div#c1042 and prepend the base URL if relative."*

Paste the chatbot's code below and verify it also gives 577 links.

In [ ]:
### Chatbot generated code here ###

## Question 4 — Fetch and display a molecular structure image from CDK Depict

CDK Depict converts a SMILES string into a 2D structure image. The URL pattern is:

```
https://www.simolecule.com/cdkdepict/depict/bow/svg?smi=<encoded_smiles>&zoom=1.0&...
```

We build it with `urllib.parse.urlencode` and fetch the SVG with `requests`.

In [ ]:
# Demo: urlencode converts special characters like '=' → '%3D'
params = {'smiles': 'CN1C=NC2=C1C(=O)N(C(=O)N2C)C'}
print(urllib.parse.urlencode(params))

In [ ]:
CDKDEPICTLINK = 'https://www.simolecule.com/cdkdepict/depict/bow'


def smiles_depict_url(smiles: str, fmt: str = 'svg') -> str:
    """Return the CDK Depict URL for the given SMILES string."""
    params = {
        'smi': smiles,
        'zoom': '1.0',
        'abbr': 'on',
        'hdisp': 'bridgehead',
        'showtitle': 'false',
        'annotate': 'none',
    }
    return f'{CDKDEPICTLINK}/{fmt}?{urllib.parse.urlencode(params)}'


def display_svg(url: str) -> None:
    """Fetch an SVG from url and render it in the notebook."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
    except RequestException as err:
        print(f'Request failed: {err}')
        return
    display(SVG(response.text))


# Try it out — diethyl phthalate
smiles = 'CCOC(=O)C1=CC=CC=C1C(=O)OCC'
url = smiles_depict_url(smiles)
print('URL:', url)
display_svg(url)

In [ ]:
# Another example — caffeine
display_svg(smiles_depict_url('CN1C=NC2=C1C(=O)N(C(=O)N2C)C'))

---
All done! To run the exercises yourself, open `09_exercise.ipynb` and work through it — come back here to check your answers.